# Hybrid boundary diagnostics

Hybrid mode is a local edge-coefficient staircase fallback. It preserves the exact continuous geometry; it does not edit or inflate polygons. This notebook prepares diagnostics and deliberately makes no claim that a solve completed.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/fdtdmesh').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
import matplotlib.pyplot as plt
import fdtdmesh as fd
from fdtdmesh.boundary import BoundaryPolicy
from fdtdmesh.solver.conformal import UnresolvedGeometryError
OUT = Path('artifacts/notebooks')
OUT.mkdir(parents=True, exist_ok=True)
RUN_GPU = False


## Exact sharp tip and narrow gap

The polygon has a sharp tip facing a nearby rectangle. Both shapes remain analytic geometry objects in SI metres.

In [ ]:
lam = fd.C0 / 1e9
vertices = [(2.25*lam, 2.2*lam), (3.0*lam, 3.0*lam), (2.25*lam, 3.8*lam), (2.45*lam, 3.0*lam)]
def make_sim(boundary):
    s = fd.Simulation(fmin=0.95e9, fmax=1.05e9, solver=fd.SolverSettings(dft_bins=1), boundary=boundary)
    s.add_polygon(vertices, name='sharp-tip')
    s.add_rectangle((3.08*lam, 3.42*lam), (2.72*lam, 3.28*lam), name='narrow-gap-block')
    return s
sim = make_sim(BoundaryPolicy(mode='hybrid'))
sim.apply_mesh('quasi_uniform', cells=(120, 120))
sim.plot_geometry(mesh=True, units='wavelength', figsize=(16,14)).savefig(OUT / '20_exact_polygon.png', dpi=120, bbox_inches='tight')
sim.plot_discretization(units='wavelength', show_fallback=True, figsize=(16,14)).savefig(OUT / '21_hybrid_fallbacks.png', dpi=120, bbox_inches='tight')
diagnostics = sim.discretization.boundary_report
print({key: diagnostics[key] for key in ('fallback_cells', 'expanded_cells', 'fallback_area_fraction', 'reasons')})


## Strict conformal comparison

A strict attempt is useful as a diagnostic: unresolved topology is reported explicitly, while the exact scene is retained.

In [ ]:
strict = make_sim(BoundaryPolicy(mode='conformal', on_fallback='error'))
try:
    strict.apply_mesh('custom', mesh=sim.mesh)
    print('strict mesh accepted at 120 × 120')
except UnresolvedGeometryError as exc:
    print('strict mesh unresolved:', exc)


## Optional GPU comparison

Set `RUN_GPU=True` only on a CUDA-enabled environment. The two budgets below are comparison inputs; this example does not assert or report completed results.

In [ ]:
if RUN_GPU:
    fields = []
    for budget in ((120, 120), (180, 180)):
        trial = make_sim(BoundaryPolicy(mode='hybrid'))
        trial.apply_mesh('quasi_uniform', cells=budget)
        result = trial.solve(progress=True)
        print('completed budget:', budget, 'status:', result.diagnostics.get('status'))
        result.save(OUT / f'hybrid_{budget[0]}.h5')
        fields.append(result)
    import numpy as np
    difference = np.linalg.norm(fields[0].far_field-fields[1].far_field)/np.linalg.norm(fields[1].far_field)
    print('Consecutive-budget complex difference (not exact error):', difference)
